In [ ]:
# Cell 1 — Imports and Constants
print('Cell 1: Imports and Constants')

import os, csv, time, random, subprocess
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.cuda.amp as amp                     # AMP: autocast + GradScaler
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.models import DenseNet121_Weights

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.utils import resample
from sklearn.model_selection import train_test_split

# ── Hyper-parameters ──────────────────────────────────────────────────────────
SEED               = 42
BATCH_SIZE         = 64       # doubled: saturates T4 VRAM better
NUM_WORKERS        = 4        # doubled: prevents data-loading CPU bottleneck
PREFETCH_FACTOR    = 4
EPOCHS_1           = 5
EPOCHS_2           = 15
LR_1               = 1e-3
LR_2               = 1e-5
PATIENCE           = 5
NUM_CLASSES        = 2
CLASSES            = ['Benign', 'Malignant']
LABEL2IDX          = {'Benign': 0, 'Malignant': 1}
IMAGENET_MEAN      = [0.485, 0.456, 0.406]
IMAGENET_STD       = [0.229, 0.224, 0.225]
IMAGE_SIZE         = 224
TARGET_SENSITIVITY = 0.90
USE_AMP            = True     # Tensor Core mixed-precision (FP16 forward, FP32 master weights)

# ── Paths ─────────────────────────────────────────────────────────────────────
ISIC_DIR        = Path('/kaggle/working/isic_data')
SPLITS_DIR      = Path('/kaggle/working/splits')
WORKING_DIR     = Path('/kaggle/working')
CHECKPOINT_DIR  = WORKING_DIR / 'checkpoints'
BEST_MODEL_PATH = WORKING_DIR / 'best_model.pth'

SPLITS_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('GPUs:',   torch.cuda.device_count())
print('AMP enabled:', USE_AMP and DEVICE.type == 'cuda')


In [ ]:
# Cell 2 — Download Data
print('Cell 2: Download Data')

# Install ISIC CLI
pip_result = subprocess.run(
    ['pip', 'install', 'isic-cli', '-q'],
    capture_output=True, text=True
)
if pip_result.returncode != 0:
    print('pip stderr:', pip_result.stderr[-400:])
else:
    print('isic-cli installed (or already present)')

os.makedirs('/kaggle/working/isic_data', exist_ok=True)

# Download images — subprocess.run captures all errors
dl_result = subprocess.run(
    ['isic', 'image', 'download', '--limit', '10000',
     '/kaggle/working/isic_data'],
    capture_output=True, text=True
)
if dl_result.stdout:
    print(dl_result.stdout[-1000:])
if dl_result.returncode != 0:
    print('Download stderr:', dl_result.stderr[-500:])

print('Download complete')


In [ ]:
# Cell 3 — Verify Data
print('Cell 3: Verify Data')

# Count images on disk
img_files = list(ISIC_DIR.glob('*.jpg')) + list(ISIC_DIR.glob('*.png'))
print(f'Total images on disk: {len(img_files)}')

# Load metadata
meta_path = ISIC_DIR / 'metadata.csv'
if not meta_path.exists():
    raise RuntimeError(f'metadata.csv not found at {meta_path}')

meta_df = pd.read_csv(meta_path)
print(f'Metadata shape : {meta_df.shape}')
print(f'Columns        : {list(meta_df.columns)}')

# Confirm diagnosis_3 exists
if 'diagnosis_3' not in meta_df.columns:
    raise RuntimeError(
        f"'diagnosis_3' column not found. Available: {list(meta_df.columns)}"
    )

print('\ndiagnosis_3 value counts:')
print(meta_df['diagnosis_3'].value_counts().to_string())

# Count malignant cases (everything that is NOT Nevus)
malignant_count = (meta_df['diagnosis_3'] != 'Nevus').sum()
print(f'\nMalignant cases (non-Nevus): {malignant_count}')

if malignant_count < 500:
    raise RuntimeError(
        f'Insufficient malignant cases: {malignant_count} < 500. '
        'Re-run download with a higher --limit.'
    )

print('Verification passed!')


In [ ]:
# Cell 4 — Data Loading with Patient-Level Splitting
print('Cell 4: Data Loading with Patient-Level Splitting')

# ── Load metadata ─────────────────────────────────────────────────────────────
meta_path = ISIC_DIR / 'metadata.csv'
meta_df   = pd.read_csv(meta_path)

# Identify the isic_id column (fallback to first column)
_isic_id_col = 'isic_id' if 'isic_id' in meta_df.columns else meta_df.columns[0]

# ── Binary label mapping: Nevus → Benign (0), everything else → Malignant (1) ─
meta_df['label_idx']  = (meta_df['diagnosis_3'] != 'Nevus').astype(int)
meta_df['label_name'] = meta_df['label_idx'].map({0: 'Benign', 1: 'Malignant'})

# ── Construct image paths ─────────────────────────────────────────────────────
meta_df['image_path'] = meta_df[_isic_id_col].apply(
    lambda x: str(ISIC_DIR / f'{x}.jpg')
)

# ── Filter to images that actually exist on disk ───────────────────────────────
valid_mask = meta_df['image_path'].apply(lambda p: Path(p).exists())
missing    = (~valid_mask).sum()
if missing:
    print(f'Warning: {missing} image paths not found on disk — skipping.')
meta_df = meta_df[valid_mask].reset_index(drop=True)
print(f'Valid rows after path verification: {len(meta_df)}')

# ── Patient-level split helper ────────────────────────────────────────────────
def _stratified_split_list(lst, r1=0.70, r2=0.15, seed=SEED):
    rng = random.Random(seed)
    lst = list(lst)
    rng.shuffle(lst)
    n  = len(lst)
    i1 = int(n * r1)
    i2 = int(n * (r1 + r2))
    return lst[:i1], lst[i1:i2], lst[i2:]

use_patient_split = (
    'patient_id' in meta_df.columns
    and meta_df['patient_id'].notna().sum() > 0
)

if use_patient_split:
    print('Using patient-level stratified splitting.')
    patient_label = (
        meta_df.groupby('patient_id')['label_idx']
        .agg(lambda x: int(x.mode()[0]))
        .reset_index()
    )
    patient_label.columns = ['patient_id', 'label_idx']

    benign_pts    = patient_label.loc[patient_label['label_idx'] == 0, 'patient_id'].tolist()
    malignant_pts = patient_label.loc[patient_label['label_idx'] == 1, 'patient_id'].tolist()

    b_tr, b_va, b_te = _stratified_split_list(benign_pts)
    m_tr, m_va, m_te = _stratified_split_list(malignant_pts)

    train_pts = set(b_tr + m_tr)
    val_pts   = set(b_va + m_va)
    test_pts  = set(b_te + m_te)

    train_df = meta_df[meta_df['patient_id'].isin(train_pts)].copy()
    val_df   = meta_df[meta_df['patient_id'].isin(val_pts)].copy()
    test_df  = meta_df[meta_df['patient_id'].isin(test_pts)].copy()

else:
    print('No patient_id column — using image-level stratified splitting.')

    benign_df    = meta_df[meta_df['label_idx'] == 0]
    malignant_df = meta_df[meta_df['label_idx'] == 1]

    def _img_split(df, r1=0.70, r2=0.15):
        tr, tmp = train_test_split(df, test_size=(1 - r1), random_state=SEED)
        rel_val = r2 / (1 - r1)
        va, te  = train_test_split(tmp, test_size=(1 - rel_val), random_state=SEED)
        return tr, va, te

    b_tr, b_va, b_te = _img_split(benign_df)
    m_tr, m_va, m_te = _img_split(malignant_df)

    train_df = pd.concat([b_tr, m_tr]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    val_df   = pd.concat([b_va, m_va]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_df  = pd.concat([b_te, m_te]).sample(frac=1, random_state=SEED).reset_index(drop=True)

# ── Save splits ───────────────────────────────────────────────────────────────
_cols = ['image_path', 'label_idx', 'label_name']
train_df[_cols].to_csv(SPLITS_DIR / 'train.csv', index=False)
val_df[_cols].to_csv(SPLITS_DIR   / 'val.csv',   index=False)
test_df[_cols].to_csv(SPLITS_DIR  / 'test.csv',  index=False)

for _name, _df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    _b = (_df['label_idx'] == 0).sum()
    _m = (_df['label_idx'] == 1).sum()
    print(f'{_name:5s}: {len(_df):5d} total | Benign={_b} | Malignant={_m}')

print('Splits saved to', SPLITS_DIR)


In [ ]:
# Cell 5 — Dataset and Transforms
print('Cell 5: Dataset and Transforms')

# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.3, hue=0.05),
    transforms.RandomPerspective(p=0.3),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Dataset ───────────────────────────────────────────────────────────────────
class SkinLesionDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df        = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row['label_idx'])
        try:
            # Context manager ensures the file handle is closed in each worker
            with Image.open(row['image_path']) as _raw:
                img = _raw.convert('RGB')
            if self.transform:
                img = self.transform(img)
        except Exception:
            # Return a black tensor + label 0 so the batch never crashes
            img   = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
            label = 0
        return img, label

# ── DataLoader factory ────────────────────────────────────────────────────────
def make_loader(csv_path, transform, is_train=False,
                batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    dataset = SkinLesionDataset(csv_path, transform=transform)
    sampler = None

    if is_train:
        labels         = dataset.df['label_idx'].values
        class_counts   = np.bincount(labels)
        sample_weights = (1.0 / class_counts)[labels]
        sampler        = WeightedRandomSampler(
            weights     = torch.FloatTensor(sample_weights),
            num_samples = len(dataset),
            replacement = True
        )
        cw = compute_class_weight('balanced',
                                  classes=np.unique(labels), y=labels)
        print(f'  Class weights — Benign: {cw[0]:.4f}, Malignant: {cw[1]:.4f}')

    pf = PREFETCH_FACTOR if num_workers > 0 else None

    return DataLoader(
        dataset,
        batch_size         = batch_size,
        sampler            = sampler,
        shuffle            = False,
        num_workers        = num_workers,
        pin_memory         = True,
        persistent_workers = (num_workers > 0),
        prefetch_factor    = pf,
        drop_last          = is_train,   # drop uneven final batch in training only
    )

train_loader = make_loader(SPLITS_DIR / 'train.csv', train_transform, is_train=True)
val_loader   = make_loader(SPLITS_DIR / 'val.csv',   val_transform,   is_train=False)
test_loader  = make_loader(SPLITS_DIR / 'test.csv',  val_transform,   is_train=False)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')


In [ ]:
# Cell 6 — Model
print('Cell 6: Model')

def build_model():
    """Return a DenseNet121 with a custom 2-class head."""
    net = models.densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
    in_features = net.classifier.in_features  # 1024
    net.classifier = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, NUM_CLASSES),
    )
    return net

model = build_model()

if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs with DataParallel')
    model = nn.DataParallel(model)

model = model.to(DEVICE)

# AMP GradScaler — no-op when CUDA is unavailable or USE_AMP=False
scaler = amp.GradScaler(enabled=(USE_AMP and DEVICE.type == 'cuda'))

def freeze_backbone(m):
    """Freeze every layer except the classifier head."""
    base = m.module if isinstance(m, nn.DataParallel) else m
    for name, param in base.named_parameters():
        param.requires_grad = ('classifier' in name)

def unfreeze_all(m):
    """Unfreeze all parameters."""
    for param in m.parameters():
        param.requires_grad = True

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'GradScaler enabled   : {scaler.is_enabled()}')


In [ ]:
# Cell 7 — Training Utilities
print('Cell 7: Training Utilities')

def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    """One full pass over loader with AMP; returns mean loss."""
    model.train()
    running_loss = torch.tensor(0.0, device=device)
    n = 0
    for imgs, labels in loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # zero_grad with set_to_none skips the memset — saves GPU bandwidth
        optimizer.zero_grad(set_to_none=True)

        with amp.autocast(enabled=scaler.is_enabled()):
            loss = criterion(model(imgs), labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate on GPU; only one .item() call at the end of the epoch
        running_loss += loss.detach() * imgs.size(0)
        n            += imgs.size(0)

    return (running_loss.item() / n) if n else 0.0


def compute_macro_auc(model, loader, device):
    """Binary AUC on the malignant-class probability. Returns 0.0 on edge cases."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs  = imgs.to(device, non_blocking=True)
            with amp.autocast(enabled=(USE_AMP and device.type == 'cuda')):
                logits = model(imgs)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.extend(labels.numpy())
    all_probs  = np.vstack(all_probs)
    all_labels = np.array(all_labels)
    if len(np.unique(all_labels)) < 2:
        return 0.0
    try:
        return float(roc_auc_score(all_labels, all_probs[:, 1]))
    except Exception:
        return 0.0


def save_checkpoint(path, epoch, val_auc, threshold, model, classes):
    base = model.module if isinstance(model, nn.DataParallel) else model
    torch.save({
        'epoch'            : epoch,
        'val_auc'          : val_auc,
        'threshold'        : threshold,
        'model_state_dict' : base.state_dict(),
        'classes'          : classes,
    }, path)


def run_phase(model, train_loader, val_loader, optimizer, scheduler,
              criterion, device, scaler, n_epochs, phase_name, log_writer,
              patience=PATIENCE):
    """Train for n_epochs with early stopping; returns best val AUC."""
    best_auc, best_epoch, no_improve = 0.0, 0, 0

    for epoch in range(1, n_epochs + 1):
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer,
                                     criterion, device, scaler)
        val_auc    = compute_macro_auc(model, val_loader, device)
        elapsed    = time.time() - t0

        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_auc)
        else:
            scheduler.step()

        print(f'[{phase_name}] Epoch {epoch:>2}/{n_epochs} | '
              f'Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f} | '
              f'Time: {elapsed:.1f}s')
        log_writer.writerow([phase_name, epoch,
                             f'{train_loss:.4f}', f'{val_auc:.4f}',
                             f'{elapsed:.1f}'])

        if val_auc > best_auc:
            best_auc, best_epoch, no_improve = val_auc, epoch, 0
            save_checkpoint(BEST_MODEL_PATH, epoch, val_auc, 0.5, model, CLASSES)
            ckpt = CHECKPOINT_DIR / f'{phase_name}_ep{epoch:03d}_auc{val_auc:.4f}.pth'
            save_checkpoint(ckpt, epoch, val_auc, 0.5, model, CLASSES)
            print(f'  -> New best saved (AUC={best_auc:.4f})')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping (no improvement for {patience} epochs)')
                break

    print(f'Phase {phase_name} done. Best AUC: {best_auc:.4f} at epoch {best_epoch}')
    return best_auc


print('Training utilities defined.')


In [ ]:
# Cell 8 — Training
print('Cell 8: Training')

criterion = nn.CrossEntropyLoss()

_log_path   = WORKING_DIR / 'training_log.csv'
_log_file   = open(_log_path, 'w', newline='')
_log_writer = csv.writer(_log_file)
_log_writer.writerow(['phase', 'epoch', 'train_loss', 'val_auc', 'time_s'])

try:
    # ── Phase 1: frozen backbone ──────────────────────────────────────────────
    print('\n=== Phase 1: Frozen Backbone ===')
    freeze_backbone(model)
    _trainable_p1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable parameters (Phase 1): {_trainable_p1:,}')

    # List comprehension (not filter iterator) so Adam gets a concrete param list
    _p1_params = [p for p in model.parameters() if p.requires_grad]
    _opt1  = torch.optim.Adam(_p1_params, lr=LR_1)
    _sch1  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        _opt1, mode='max', patience=2, factor=0.5
    )
    _best1 = run_phase(
        model, train_loader, val_loader, _opt1, _sch1, criterion,
        DEVICE, scaler, EPOCHS_1, 'phase1', _log_writer
    )

    # ── Phase 2: full fine-tune ───────────────────────────────────────────────
    print('\n=== Phase 2: Full Fine-tuning ===')
    unfreeze_all(model)
    _trainable_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable parameters (Phase 2): {_trainable_p2:,}')

    _opt2  = torch.optim.Adam(model.parameters(), lr=LR_2, weight_decay=1e-4)
    _sch2  = torch.optim.lr_scheduler.CosineAnnealingLR(_opt2, T_max=EPOCHS_2)
    _best2 = run_phase(
        model, train_loader, val_loader, _opt2, _sch2, criterion,
        DEVICE, scaler, EPOCHS_2, 'phase2', _log_writer
    )

finally:
    _log_file.close()

print(f'\nTraining complete. Best val AUC: {max(_best1, _best2):.4f}')


In [ ]:
# Cell 9 — Threshold Calibration
print('Cell 9: Threshold Calibration')

# ── Reload best checkpoint ────────────────────────────────────────────────────
_ckpt      = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
_cal_model = build_model()
_cal_model.load_state_dict(_ckpt['model_state_dict'])
if torch.cuda.device_count() > 1:
    _cal_model = nn.DataParallel(_cal_model)
_cal_model = _cal_model.to(DEVICE)
_cal_model.eval()

# ── Inference on validation set ───────────────────────────────────────────────
_val_probs_list, _val_labels_list = [], []
with torch.no_grad():
    for _imgs, _labs in val_loader:
        _imgs = _imgs.to(DEVICE, non_blocking=True)
        with amp.autocast(enabled=(USE_AMP and DEVICE.type == 'cuda')):
            _logits = _cal_model(_imgs)
        _p = torch.softmax(_logits, dim=1).cpu().numpy()
        _val_probs_list.append(_p)
        _val_labels_list.extend(_labs.numpy())

_val_probs  = np.vstack(_val_probs_list)
_val_labels = np.array(_val_labels_list)
_val_scores = _val_probs[:, 1]

# ── ROC curve ─────────────────────────────────────────────────────────────────
_fpr_v, _tpr_v, _thresholds_v = roc_curve(_val_labels, _val_scores)

# First threshold where TPR >= TARGET_SENSITIVITY
_sens_idx = np.where(_tpr_v >= TARGET_SENSITIVITY)[0]
if len(_sens_idx) > 0:
    _sens_threshold     = float(_thresholds_v[_sens_idx[0]])
    _actual_sensitivity = float(_tpr_v[_sens_idx[0]])
else:
    _sens_threshold     = 0.5
    _actual_sensitivity = 0.0
    print('Warning: TARGET_SENSITIVITY unreachable — defaulting threshold to 0.5')

# Youden's J (reference only)
_youden_idx       = int(np.argmax(_tpr_v - _fpr_v))
_youden_threshold = float(_thresholds_v[_youden_idx])

print(f"Sensitivity-targeted threshold (TPR >= {TARGET_SENSITIVITY}): "
      f"{_sens_threshold:.4f}  (actual TPR={_actual_sensitivity:.4f})")
print(f"Youden's J optimal threshold : {_youden_threshold:.4f}")

CALIBRATED_THRESHOLD = _sens_threshold
print(f"\nCalibrated threshold: {CALIBRATED_THRESHOLD:.3f} "
      f"(sensitivity=0.90 on val set)")

# ── Re-save checkpoint with calibrated threshold ──────────────────────────────
_ckpt['threshold'] = CALIBRATED_THRESHOLD
torch.save(_ckpt, BEST_MODEL_PATH)
print('Checkpoint updated with calibrated threshold.')


In [ ]:
# Cell 10 — Evaluation
print('Cell 10: Evaluation')

# ── Load model + threshold ────────────────────────────────────────────────────
_eval_ckpt           = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
CALIBRATED_THRESHOLD = _eval_ckpt.get('threshold', 0.5)

_eval_model = build_model()
_eval_model.load_state_dict(_eval_ckpt['model_state_dict'])
if torch.cuda.device_count() > 1:
    _eval_model = nn.DataParallel(_eval_model)
_eval_model = _eval_model.to(DEVICE)
_eval_model.eval()

# ── Inference on test set ─────────────────────────────────────────────────────
_test_probs_list, _test_labels_list = [], []
with torch.no_grad():
    for _imgs, _labs in test_loader:
        _imgs = _imgs.to(DEVICE, non_blocking=True)
        with amp.autocast(enabled=(USE_AMP and DEVICE.type == 'cuda')):
            _logits = _eval_model(_imgs)
        _p = torch.softmax(_logits, dim=1).cpu().numpy()
        _test_probs_list.append(_p)
        _test_labels_list.extend(_labs.numpy())

test_probs  = np.vstack(_test_probs_list)
test_labels = np.array(_test_labels_list)
test_scores = test_probs[:, 1]
test_preds  = (test_scores >= CALIBRATED_THRESHOLD).astype(int)

# ── Point estimates ───────────────────────────────────────────────────────────
test_auc       = roc_auc_score(test_labels, test_scores)
_cm            = confusion_matrix(test_labels, test_preds, labels=[0, 1])
tn, fp, fn, tp = _cm.ravel()
sensitivity    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity    = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# ── Bootstrap 95% CIs (1 000 iterations) ─────────────────────────────────────
_boot_aucs, _boot_sens, _boot_spec = [], [], []
_rng = np.random.default_rng(SEED)

for _ in range(1000):
    _idx      = _rng.integers(0, len(test_labels), len(test_labels))
    _b_labels = test_labels[_idx]
    _b_scores = test_scores[_idx]
    _b_preds  = test_preds[_idx]
    if len(np.unique(_b_labels)) < 2:
        continue
    _boot_aucs.append(roc_auc_score(_b_labels, _b_scores))
    _btn, _bfp, _bfn, _btp = confusion_matrix(
        _b_labels, _b_preds, labels=[0, 1]
    ).ravel()
    _boot_sens.append(_btp / (_btp + _bfn) if (_btp + _bfn) > 0 else 0.0)
    _boot_spec.append(_btn / (_btn + _bfp) if (_btn + _bfp) > 0 else 0.0)

auc_ci  = (np.percentile(_boot_aucs, 2.5), np.percentile(_boot_aucs, 97.5))
sens_ci = (np.percentile(_boot_sens, 2.5), np.percentile(_boot_sens, 97.5))
spec_ci = (np.percentile(_boot_spec, 2.5), np.percentile(_boot_spec, 97.5))

# ── Print results ─────────────────────────────────────────────────────────────
print(f'Binary AUC          : {test_auc:.4f}  '
      f'(95% CI: {auc_ci[0]:.4f} – {auc_ci[1]:.4f})')
print(f'Sensitivity         : {sensitivity:.4f}  '
      f'(95% CI: {sens_ci[0]:.4f} – {sens_ci[1]:.4f})')
print(f'Specificity         : {specificity:.4f}  '
      f'(95% CI: {spec_ci[0]:.4f} – {spec_ci[1]:.4f})')
print(f'Confusion matrix    : TP={tp}  TN={tn}  FP={fp}  FN={fn}')

# ── Confusion matrix plot ─────────────────────────────────────────────────────
_fig_cm, _ax_cm = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=_cm, display_labels=CLASSES).plot(
    ax=_ax_cm, colorbar=False
)
_ax_cm.set_title('Confusion Matrix (Test Set)')
_fig_cm.savefig(WORKING_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close(_fig_cm)
print('Saved confusion_matrix.png')

# ── ROC curve plot ────────────────────────────────────────────────────────────
_fpr_t, _tpr_t, _ = roc_curve(test_labels, test_scores)
_fig_roc, _ax_roc = plt.subplots(figsize=(6, 5))
_ax_roc.plot(_fpr_t, _tpr_t, lw=2, label=f'AUC = {test_auc:.4f}')
_ax_roc.plot([0, 1], [0, 1], 'k--', lw=1)
_ax_roc.set_xlabel('False Positive Rate')
_ax_roc.set_ylabel('True Positive Rate')
_ax_roc.set_title('ROC Curve (Test Set)')
_ax_roc.legend(loc='lower right')
_fig_roc.savefig(WORKING_DIR / 'roc_curve.png', dpi=150, bbox_inches='tight')
plt.close(_fig_roc)
print('Saved roc_curve.png')

# ── Subgroup analysis ─────────────────────────────────────────────────────────
_meta_eval = pd.read_csv(ISIC_DIR / 'metadata.csv')
_isic_id_c = 'isic_id' if 'isic_id' in _meta_eval.columns else _meta_eval.columns[0]
_meta_eval['image_path'] = _meta_eval[_isic_id_c].apply(
    lambda x: str(ISIC_DIR / f'{x}.jpg')
)
_test_split = pd.read_csv(SPLITS_DIR / 'test.csv')
_sub_cols   = [c for c in ['sex', 'anatom_site_general'] if c in _meta_eval.columns]

if _sub_cols:
    _merged = _test_split.merge(
        _meta_eval[['image_path'] + _sub_cols], on='image_path', how='left'
    )
    _merged['_score'] = test_scores
    _merged['_label'] = test_labels

    for _col in _sub_cols:
        print(f'\nSubgroup AUC by {_col}:')
        for _grp in sorted(_merged[_col].dropna().unique()):
            _g = _merged[_merged[_col] == _grp]
            if len(_g) < 10 or len(_g['_label'].unique()) < 2:
                print(f'  {_grp}: insufficient data (n={len(_g)})')
                continue
            _g_auc = roc_auc_score(_g['_label'], _g['_score'])
            print(f'  {_grp}: AUC={_g_auc:.4f} (n={len(_g)})')
else:
    print('No sex / anatom_site_general columns found — skipping subgroup analysis.')


In [ ]:
# Cell 11 — Save Outputs
print('Cell 11: Save Outputs')

# ── Reload threshold from checkpoint ─────────────────────────────────────────
_final_ckpt          = torch.load(BEST_MODEL_PATH, map_location='cpu', weights_only=False)
CALIBRATED_THRESHOLD = _final_ckpt.get('threshold', 0.5)
_final_val_auc       = _final_ckpt.get('val_auc', 0.0)

print(f'\nCalibrated threshold : {CALIBRATED_THRESHOLD:.4f}')
print(f'Best model path      : {BEST_MODEL_PATH}')

# ── List all files saved to /kaggle/working/ ──────────────────────────────────
print('\nFiles saved to /kaggle/working/:')
for _f in sorted(WORKING_DIR.glob('*')):
    if _f.is_file():
        _kb = _f.stat().st_size / 1024
        print(f'  {_f.name:<35s} ({_kb:>8.1f} KB)')

# ── Reload split counts ───────────────────────────────────────────────────────
_tr_df = pd.read_csv(SPLITS_DIR / 'train.csv')
_va_df = pd.read_csv(SPLITS_DIR / 'val.csv')
_te_df = pd.read_csv(SPLITS_DIR / 'test.csv')
_total = len(_tr_df) + len(_va_df) + len(_te_df)

# ── Summary card ──────────────────────────────────────────────────────────────
print('\n' + '=' * 52)
print('  DERMOSCAN — TRAINING SUMMARY CARD')
print('=' * 52)
print(f'  Dataset size         : {_total}')
print(f'  Train count          : {len(_tr_df)}')
print(f'  Val count            : {len(_va_df)}')
print(f'  Test count           : {len(_te_df)}')
print(f'  Best val AUC         : {_final_val_auc:.4f}')
print(f'  Test AUC (95% CI)    : {test_auc:.4f}  '
      f'({auc_ci[0]:.4f} – {auc_ci[1]:.4f})')
print(f'  Sensitivity (95% CI) : {sensitivity:.4f}  '
      f'({sens_ci[0]:.4f} – {sens_ci[1]:.4f})')
print(f'  Specificity (95% CI) : {specificity:.4f}  '
      f'({spec_ci[0]:.4f} – {spec_ci[1]:.4f})')
print(f'  Calibrated threshold : {CALIBRATED_THRESHOLD:.4f}')
print('=' * 52)
